In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm
import scipy.io as sio
from scipy import stats
from importlib import reload
from matplotlib import pyplot as plt
import sys
sys.path.append(r'C:\\Users\\scanimage\\Documents\\JJM\\post_cnmfe_analysis')
#sys.path.append('/Users/johnmarshall/Documents/Analysis/PythonAnalysisScripts/post_cmfe_analysis')
import os
os.chdir(r'C:\\Users\\scanimage\\Documents\\JJM\\post_cnmfe_analysis')
import python_utils_jjm as utils_jjm
import python_utils_jjm as utils_jjm
import dlc_utils
from sklearn.preprocessing import MinMaxScaler
import scipy.spatial.distance as dist
import itertools
import math
import warnings
%matplotlib inline
from matplotlib import pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML
#import av
from multiprocessing import Pool
import functools
import glob
#plt.rcParams['animation.ffmpeg_path']='/home/jma819/.conda/envs/caiman/bin/ffmpeg'
warnings.filterwarnings(action='once')

C:\Users\scanimage\AppData\Local\Temp\ipykernel_12300\152334368.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
## helper functions for analysis 

#alignment with ezTrack location tracking data

def alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate):
    #'\\'.join(sessionPath.split(os.sep)[:-1])+'\\timestamp.dat'
    print(sessionPath.split(os.sep)[:-1])
    # load eZ track output and behavior camera timestamps from miniscope software 
    ezTrackOutput = pd.read_csv(sessionPath)
    timestampfile = pd.read_table('\\'.join(sessionPath.split(os.sep)[:-1])+'\\timeStamps.csv', delimiter=',')
    miniscope_timestampfile = pd.read_table('\\'.join(sessionPath.split(os.sep)[:-1])+'\\timeStampsMiniscope.csv', delimiter=',')
    
    timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
    miniscopetimestamp_td = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)
    
    behavCam_frames = []
    sys_clock_behavCam = []
    #create "key" for aligning miniscope frames to timestamp file
    #then create behavior TD and align
    for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td['Frame Number']))):
        #get sys clock time of each miniscope recorded frame
        #sys_clock_msCam = time_stamps['sysClock'].loc[msCam_frame]
        #find behav cam frame closest to sys clock time of ms frame
        behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
        #this is the behavCamIndex that is closest to the corresponding miniscope frame 
        behavCam_frames.append(behavCam_frame)
        sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

    behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
    #ezTrackOutput

    miniscopetimestamp_td['closestBehavCamFrameIdx'] = behavCamIdxToAlign

    X_coor=[]
    Y_coor=[]
    Distance_px=[] 

    for i in miniscopetimestamp_td['closestBehavCamFrameIdx'].values:
        X_coor.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['X'])
        Y_coor.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['Y'])
        Distance_px.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['Distance_px'])
    
    miniscopetimestamp_td['X_coor'] = X_coor
    miniscopetimestamp_td['Y_coor'] = Y_coor
    miniscopetimestamp_td['Distance_px'] = Distance_px
    
    return(miniscopetimestamp_td)

## load and do some preprocessing on the CNMFE traces 
def getCellTraces(dir_path, file_name):
    
    CNMFE_file = dir_path+file_name
    ## load cnmfe output and perform some basic adjustments (normalize traces to peak intensity, calculate z score)
    cell_fluorescence = sio.loadmat(CNMFE_file)

    C_timedelta = utils_jjm.create_fluorescence_time_delta(cell_fluorescence['C'])
    C_normalized = C_timedelta.apply(utils_jjm.normalize).set_index(pd.to_timedelta(np.linspace(0, (len(C_timedelta)-1)*(1/20), len(C_timedelta)), unit='s'), drop=True)
    C_z_scored = C_timedelta.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_timedelta)-1)*(1/20), len(C_timedelta)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/20), len(C_normalized)), unit='s'), drop=True)
    C_normalized_z_scored.drop('msCamFrame', axis=1, inplace=True)

    spatial_components=np.array(cell_fluorescence['A'].todense())

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
    com_df, spatial_components = utils_jjm.return_spatial_info(CNMFE_file, 0.6, dims=(600, 600))
    cell_contours, for_dims = utils_jjm.create_contour_layouts(spatial_components, dims=(600, 600))

    C_normalized_z_scored.to_csv(dir_path+file_name.strip("out.mat")+'_C_traces_filtered_origHz.csv')
    com_df.to_csv(dir_path+file_name.strip("out.mat")+'_com_filtered.csv')
    
    print('finished, saved:')
    print(file_name)
    
    return(C_normalized_z_scored, com_df)

In [41]:
#behavior analysis info
savePath = r'F:\\JJM\\miniscope_analysis_CA1\\prelimAnalysis\\'
sessionPath = r'F:\\JJM\\miniscope_analysis_CA1\\prelimAnalysis\\15_42_30_21925_BehaviorCamera\\concactenated_behavCam00behavCam08_LocationOutput.csv'
behavCamFrameRate = 15
miniscopeCamFrameRate = 20 

#cnmfe info 
dir_path = r'F:\\JJM\\miniscope_analysis_CA1\\prelimAnalysis\\15_42_30_21925_CNMFE\\'
file_name = '22-Sep_11_11_55_out.mat'


behavCamDataAligned = alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate)

C_normalized_z_scored, com = getCellTraces(dir_path, file_name)

['F:', '', 'JJM', '', 'miniscope_analysis_CA1', '', 'prelimAnalysis', '', '15_42_30_21925_BehaviorCamera', '']


100%|██████████████████████████████████████████████████████████████████████████| 11859/11859 [00:08<00:00, 1386.26it/s]


finished, saved:
22-Sep_11_11_55_out.mat


In [42]:
#display cnmfe data 
C_normalized_z_scored.head()

,1,2,3,4,5,6,7,8,9,10,...,121,122,123,124,125,126,127,128,129,130
0 days 00:00:00,-0.121345,-0.594228,-0.84946,-0.597868,0.193064,0.132245,-0.170238,0.578896,1.747647,-0.473136,...,1.130391,-0.776492,0.102906,-0.579315,0.761962,1.555160,5.382886,1.105452,-0.593538,-0.743685
0 days 00:00:00.050000,-0.128869,-0.601092,-0.84946,-0.597868,0.178674,0.119333,-0.181570,0.555613,1.698388,-0.473136,...,1.076278,-0.776492,0.085719,-0.579315,0.712719,1.437189,5.296520,1.061107,-0.593538,-0.743685
0 days 00:00:00.100000,-0.136255,-0.607823,-0.84946,-0.597868,0.164474,0.106665,-0.192701,0.532667,1.650202,-0.473136,...,1.023549,-0.776492,0.068830,-0.579315,0.665258,1.325776,5.211389,1.017707,-0.593538,-0.743685
0 days 00:00:00.150000,-0.143506,-0.614424,-0.84946,-0.597868,0.150461,0.094237,-0.203635,0.510051,1.603064,-0.473136,...,0.972167,-0.776492,0.052233,-0.579315,0.619513,1.220554,5.127475,0.975233,-0.593538,-0.743685
0 days 00:00:00.200000,-0.150623,-0.620898,-0.84946,-0.597868,0.136632,0.082045,-0.214375,0.487762,1.556951,-0.473136,...,0.922099,-0.776492,0.035924,-0.579315,0.575423,1.121181,5.044760,0.933663,-0.593538,-0.743685


In [43]:
#tracking data 
behavCamDataAligned.head()

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0 days 00:00:00,0,-41,0,0,197.317296,17.354612,0.000000
0 days 00:00:00.050000,1,11,0,1,201.492751,16.617938,4.239943
0 days 00:00:00.100000,2,61,0,2,205.888295,16.356386,4.403319
0 days 00:00:00.150000,3,110,0,2,205.888295,16.356386,4.403319
0 days 00:00:00.200000,4,161,0,3,205.888295,16.356386,4.403319


In [44]:
# align tracking data to CNMFE for the movies we've analyzed 
CNMFE_aligned = pd.concat([C_normalized_z_scored, behavCamDataAligned.iloc[0:len(C_normalized_z_scored)]], axis=1)
CNMFE_aligned.to_csv(dir_path+file_name.strip("out.mat")+'cellTracesAlignedToTracking.csv')

In [50]:
# for each cell find the peaks above threshold, then get the x,y coordinates of the mouse at those peaks
activity_threshold = 2.5 

cellFiringCoordinates = {}
for cell in list(C_normalized_z_scored.columns):
    coordinatesAtPeak = CNMFE_aligned[['Frame Number', 'closestBehavCamFrameIdx', 'Time Stamp (ms)', 'Y_coor', 'X_coor']].loc[C_normalized_z_scored[cell]>activity_threshold]
    coordinatesAtPeak.to_csv(dir_path+file_name.strip("out.mat")+'cell_'+str(cell)+'_coordinatesabove_'+str(activity_threshold)+'_Zscore_.csv')
    cellFiringCoordinates[cell] = coordinatesAtPeak
    

In [51]:
CNMFE_aligned.head()

,1,2,3,4,5,6,7,8,9,10,...,128,129,130,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0 days 00:00:00,-0.121345,-0.594228,-0.84946,-0.597868,0.193064,0.132245,-0.170238,0.578896,1.747647,-0.473136,...,1.105452,-0.593538,-0.743685,0,-41,0,0,197.317296,17.354612,0.000000
0 days 00:00:00.050000,-0.128869,-0.601092,-0.84946,-0.597868,0.178674,0.119333,-0.181570,0.555613,1.698388,-0.473136,...,1.061107,-0.593538,-0.743685,1,11,0,1,201.492751,16.617938,4.239943
0 days 00:00:00.100000,-0.136255,-0.607823,-0.84946,-0.597868,0.164474,0.106665,-0.192701,0.532667,1.650202,-0.473136,...,1.017707,-0.593538,-0.743685,2,61,0,2,205.888295,16.356386,4.403319
0 days 00:00:00.150000,-0.143506,-0.614424,-0.84946,-0.597868,0.150461,0.094237,-0.203635,0.510051,1.603064,-0.473136,...,0.975233,-0.593538,-0.743685,3,110,0,2,205.888295,16.356386,4.403319
0 days 00:00:00.200000,-0.150623,-0.620898,-0.84946,-0.597868,0.136632,0.082045,-0.214375,0.487762,1.556951,-0.473136,...,0.933663,-0.593538,-0.743685,4,161,0,3,205.888295,16.356386,4.403319


In [52]:
#view trace for cell 
#cell = 1
#plt.plot(CNMFE_aligned[cell])

In [53]:
#plt.plot(CNMFE_aligned['X_coor'])

In [55]:
cellFiringCoordinates[1]

,Frame Number,closestBehavCamFrameIdx,Time Stamp (ms),Y_coor,X_coor
0 days 00:00:04.400000,88,64,4412,17.151296,350.811525
0 days 00:00:04.450000,89,65,4464,17.151296,350.811525
0 days 00:00:04.500000,90,65,4514,17.151296,350.811525
0 days 00:00:15.700000,314,228,15850,25.645720,462.789562
0 days 00:00:15.750000,315,229,15901,25.645720,462.789562
...,...,...,...,...,...
0 days 00:04:14.050000,5081,3686,257096,11.658002,22.511453
0 days 00:04:14.100000,5082,3687,257148,11.739276,23.209192
0 days 00:04:14.150000,5083,3688,257198,11.860120,23.778928
0 days 00:04:14.200000,5084,3689,257248,11.860120,23.778928
